# Implicit ALS experiments

This notebook evaluates and tunes the implicit-feedback ALS retriever.

The experiments assess both standalone candidate retrieval quality and the retriever's contribution to the hybrid candidate pool. Model selection is performed on the validation period while the final test split remains untouched.

## Goals

- evaluate the contribution of implicit status and negative-rating signals;
- study latent factor dimensionality;
- tune regularization and training iterations;
- measure the effect of ALS changes on hybrid candidate recall;
- select a final ALS configuration for subsequent system experiments.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from shiki_recsys.config.settings import get_settings
from shiki_recsys.config.training import (
    ALSSignalConfidencesConfig,
    load_training_config,
)
from shiki_recsys.database.session import (
    create_database_engine,
    create_session_factory,
)
from shiki_recsys.evaluation.retrievers import (
    RetrieverCandidateRun,
    evaluate_retriever_run,
    generate_retriever_evaluation_run,
)
from shiki_recsys.features.als_signals import build_als_signed_interactions
from shiki_recsys.preprocessing.known_items import build_known_items
from shiki_recsys.preprocessing.splitting import chronological_split
from shiki_recsys.retrievers.common import RetrieverName
from shiki_recsys.retrievers.implicit_als import ImplicitALSRetriever
from shiki_recsys.training.data import load_training_data
from shiki_recsys.training.retrievers import fit_retrievers

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)

## Evaluation setup

The chronological split and candidate-generation pipeline are reused from the production code.

ALS variants are trained on the outer training split and evaluated against positive interactions from the validation split. The other retrievers remain fixed at their selected configuration so that changes in hybrid candidate quality can be attributed to the ALS variant being tested.

The test split is not used during model selection.

In [2]:
config = load_training_config(PROJECT_ROOT / "configs" / "training.yaml")
settings = get_settings()

engine = create_database_engine(settings)
session_factory = create_session_factory(engine)

try:
    with session_factory() as session:
        data = load_training_data(session)
finally:
    engine.dispose()

interactions = data.interactions
catalog = data.catalog

split_config = config.dataset.split

split = chronological_split(
    interactions,
    validation_fraction=split_config.validation_fraction,
    test_fraction=split_config.test_fraction,
    min_interactions_per_user=split_config.min_interactions_per_user,
)

pd.Series(
    {
        "train_interactions": len(split.train),
        "validation_interactions": len(split.validation),
        "test_interactions": len(split.test),
        "catalog_items": len(catalog),
        "retrieval_k": config.candidate_generation.retrieval_k,
        "positive_rating_threshold": (config.target.positive_rating_threshold),
    },
    name="value",
)

train_interactions           875697.0
validation_interactions      108915.0
test_interactions            108915.0
catalog_items                 23802.0
retrieval_k                     100.0
positive_rating_threshold         8.0
Name: value, dtype: float64

In [3]:
baseline_signal_confidences = ALSSignalConfidencesConfig(
    rating_8_10=2.0,
    watching=1.0,
    rewatching=1.0,
    completed=0.5,
    planned=0.5,
    on_hold=0.0,
    rating_4_5=-1.0,
    rating_1_3=-2.0,
)

baseline_als_config = config.retrievers.implicit_als.model_copy(
    update={
        "signal_confidences": baseline_signal_confidences,
        "factors": 96,
        "regularization": 0.1,
        "alpha": 1.0,
        "iterations": 40,
    }
)

experiment_config = config.model_copy(
    update={
        "retrievers": config.retrievers.model_copy(
            update={
                "implicit_als": baseline_als_config,
            }
        )
    }
)

In [4]:
(
    popularity,
    explicit_svd,
    implicit_als,
    content_tfidf,
) = fit_retrievers(
    interactions=split.train,
    catalog=catalog,
    config=experiment_config,
)

In [5]:
candidate_count = experiment_config.candidate_generation.retrieval_k

baseline_run = generate_retriever_evaluation_run(
    history_interactions=split.train,
    target_interactions=split.validation,
    positive_rating_threshold=(config.target.positive_rating_threshold),
    candidate_count=candidate_count,
    popularity=popularity,
    explicit_svd=explicit_svd,
    implicit_als=implicit_als,
    content_tfidf=content_tfidf,
)

## Model and signal experiments

ALS uses weighted implicit signals rather than explicit rating prediction. Before tuning model capacity, the contribution of different interaction types is evaluated.

The experiments progressively add status-based positive signals and explicit negative signals while keeping the ALS model hyperparameters fixed at their baseline values.

In [6]:
als_config = experiment_config.retrievers.implicit_als
known_items_by_user = build_known_items(split.train)


def evaluate_als_variant(
    *,
    signal_confidences: ALSSignalConfidencesConfig | None = None,
    factors: int | None = None,
    regularization: float | None = None,
    alpha: float | None = None,
    iterations: int | None = None,
) -> dict[str, float | int]:
    """Evaluate one ALS configuration in the hybrid candidate pool."""
    confidences = (
        als_config.signal_confidences
        if signal_confidences is None
        else signal_confidences
    )

    signed_interactions = build_als_signed_interactions(
        split.train,
        rating_8_10_confidence=confidences.rating_8_10,
        watching_confidence=confidences.watching,
        rewatching_confidence=confidences.rewatching,
        completed_confidence=confidences.completed,
        planned_confidence=confidences.planned,
        on_hold_confidence=confidences.on_hold,
        rating_4_5_confidence=confidences.rating_4_5,
        rating_1_3_confidence=confidences.rating_1_3,
    )

    retriever = ImplicitALSRetriever(
        factors=als_config.factors if factors is None else factors,
        regularization=(
            als_config.regularization if regularization is None else regularization
        ),
        alpha=als_config.alpha if alpha is None else alpha,
        iterations=(als_config.iterations if iterations is None else iterations),
        random_state=config.random_seed,
    ).fit(signed_interactions)

    candidates_by_user = {}

    for user_id, baseline_candidates in baseline_run.candidates_by_user.items():
        als_candidates = retriever.retrieve(
            user_id=user_id,
            candidate_count=candidate_count,
            exclude_anime_ids=known_items_by_user.get(user_id, set()),
        )

        candidates_by_user[user_id] = {
            **baseline_candidates,
            RetrieverName.IMPLICIT_ALS: als_candidates,
        }

    run = RetrieverCandidateRun(
        positive_items_by_user=baseline_run.positive_items_by_user,
        candidates_by_user=candidates_by_user,
    )

    evaluation = evaluate_retriever_run(
        run,
        candidate_count=candidate_count,
        catalog_size=len(catalog),
    )

    als_metrics = evaluation.individual_metrics.set_index("source").loc[
        RetrieverName.IMPLICIT_ALS.value
    ]

    contribution = next(
        item
        for item in evaluation.contributions
        if item.source == RetrieverName.IMPLICIT_ALS
    )

    return {
        "training_signals": len(signed_interactions),
        "positive_signals": int((signed_interactions["confidence"] > 0).sum()),
        "negative_signals": int((signed_interactions["confidence"] < 0).sum()),
        "supported_items": len(retriever.supported_anime_ids),
        "als_recall_at_100": als_metrics["recall_at_k"],
        "als_catalog_coverage_at_100": (als_metrics["catalog_coverage_at_k"]),
        "union_recall": evaluation.union_recall,
        "als_marginal_recall": contribution.marginal_recall,
        "als_unique_relevant_hits": contribution.unique_relevant_hits,
    }

### Signal composition

The baseline ALS representation combines strong positive ratings, weaker status-based positive signals, and explicit negative ratings.

Three progressively richer representations are compared:

1. positive ratings only;
2. positive ratings and status signals;
3. the full signed representation including negative ratings.

This ablation isolates whether the additional implicit and negative signals provide useful information for candidate generation.

In [7]:
baseline_confidences = als_config.signal_confidences

signal_variants = {
    "positive_ratings_only": baseline_confidences.model_copy(
        update={
            "watching": 0.0,
            "rewatching": 0.0,
            "completed": 0.0,
            "planned": 0.0,
            "on_hold": 0.0,
            "rating_4_5": 0.0,
            "rating_1_3": 0.0,
        }
    ),
    "positive_ratings_and_statuses": baseline_confidences.model_copy(
        update={
            "rating_4_5": 0.0,
            "rating_1_3": 0.0,
        }
    ),
    "full_signed": baseline_confidences,
}

als_signal_results = []

for variant_name, signal_confidences in signal_variants.items():
    result = evaluate_als_variant(
        signal_confidences=signal_confidences,
    )

    als_signal_results.append(
        {
            "variant": variant_name,
            **result,
        }
    )

als_signal_results = pd.DataFrame(als_signal_results)

als_signal_results

,variant,training_signals,positive_signals,negative_signals,supported_items,als_recall_at_100,als_catalog_coverage_at_100,union_recall,als_marginal_recall,als_unique_relevant_hits
0,positive_ratings_only,232153,232153,0,9598,0.328083,0.134358,0.537279,0.094482,1785
1,positive_ratings_and_statuses,558924,558924,0,21534,0.328422,0.159482,0.542579,0.099782,1729
2,full_signed,629576,558924,70652,21619,0.331908,0.158936,0.543957,0.101160,1742


#### Selection

Status-based implicit signals provide a clear benefit to the hybrid candidate generator. Compared with using positive ratings alone, adding status signals substantially increases catalog support and coverage while improving union recall from 0.5373 to 0.5426.

Adding explicit negative-rating signals provides a smaller but consistent additional improvement. The full signed representation achieves the highest standalone ALS Recall@100, union recall, and marginal recall contribution among the evaluated variants.

The positive-ratings-only variant produces a slightly larger raw number of uniquely retrieved relevant user-item pairs. However, this count is not normalized per user and does not translate into higher macro recall.

The full signed interaction representation is therefore retained for subsequent ALS experiments. Status signals account for most of the improvement over ratings-only training, while negative ratings provide a smaller additional gain.

### Status signal strength

The previous ablation shows that status-based interactions provide a meaningful improvement over rating-only ALS training.

Rather than tuning each status confidence independently, their relative weighting is kept fixed and the overall strength of the status group is varied. This preserves the intended distinction between stronger `watching` / `rewatching` signals and weaker `completed` / `planned` signals while keeping the experiment interpretable.

Explicit positive and negative rating confidences remain unchanged.

In [8]:
status_multipliers = [0.0, 0.5, 1.0, 1.5, 2.0]

als_status_strength_results = []

for multiplier in status_multipliers:
    signal_confidences = baseline_confidences.model_copy(
        update={
            "watching": baseline_confidences.watching * multiplier,
            "rewatching": baseline_confidences.rewatching * multiplier,
            "completed": baseline_confidences.completed * multiplier,
            "planned": baseline_confidences.planned * multiplier,
            "on_hold": baseline_confidences.on_hold * multiplier,
        }
    )

    result = evaluate_als_variant(
        signal_confidences=signal_confidences,
    )

    als_status_strength_results.append(
        {
            "status_multiplier": multiplier,
            "watching_confidence": signal_confidences.watching,
            "completed_confidence": signal_confidences.completed,
            **result,
        }
    )

als_status_strength_results = pd.DataFrame(als_status_strength_results)

als_status_strength_results

,status_multiplier,watching_confidence,completed_confidence,training_signals,positive_signals,negative_signals,supported_items,als_recall_at_100,als_catalog_coverage_at_100,union_recall,als_marginal_recall,als_unique_relevant_hits
0,0.0,0.0,0.00,302805,232153,70652,15302,0.326694,0.134989,0.537166,0.094369,1784
1,0.5,0.5,0.25,629576,558924,70652,21619,0.333047,0.149399,0.543879,0.101082,1770
2,1.0,1.0,0.50,629576,558924,70652,21619,0.331908,0.158936,0.543957,0.101160,1742
3,1.5,1.5,0.75,629576,558924,70652,21619,0.327188,0.166373,0.542018,0.099221,1707
4,2.0,2.0,1.00,629576,558924,70652,21619,0.322326,0.171204,0.539646,0.096849,1696


#### Selection

Status-based signals are clearly useful for ALS candidate generation. Removing them reduces union recall from 0.5440 to 0.5372 and substantially weakens the retriever's marginal contribution.

Moderate status weights perform best. Multipliers of 0.5 and 1.0 produce nearly identical hybrid recall, while stronger weighting increasingly favors catalog coverage at the expense of relevant candidate retrieval.

The baseline multiplier of 1.0 provides the highest union and marginal recall among the tested values, although the difference from 0.5 is very small. It is therefore retained rather than treated as a sharply identified optimum.

Higher status multipliers are not selected because the additional coverage does not compensate for the decline in hybrid candidate recall.

### Individual status contribution

The previous experiment shows that status-based interactions are useful as a group, but it does not establish whether each status contributes useful information.

Each non-zero status signal is therefore removed from the full signed representation one at a time while all other confidences remain unchanged.

This ablation is used to identify which status types should be retained and which weights require further tuning before the ALS model hyperparameters are optimized.

In [9]:
status_ablation_updates = {
    "full_statuses": {},
    "without_watching": {"watching": 0.0},
    "without_rewatching": {"rewatching": 0.0},
    "without_completed": {"completed": 0.0},
    "without_planned": {"planned": 0.0},
}

als_status_ablation_results = []

for variant_name, updates in status_ablation_updates.items():
    signal_confidences = baseline_confidences.model_copy(update=updates)

    result = evaluate_als_variant(
        signal_confidences=signal_confidences,
    )

    als_status_ablation_results.append(
        {
            "variant": variant_name,
            **result,
        }
    )

als_status_ablation_results = pd.DataFrame(als_status_ablation_results)

als_status_ablation_results

,variant,training_signals,positive_signals,negative_signals,supported_items,als_recall_at_100,als_catalog_coverage_at_100,union_recall,als_marginal_recall,als_unique_relevant_hits
0,full_statuses,629576,558924,70652,21619,0.331908,0.158936,0.543957,0.101160,1742
1,without_watching,617240,546588,70652,21614,0.323918,0.156205,0.539898,0.097101,1737
2,without_rewatching,628731,558079,70652,21619,0.330306,0.158474,0.543738,0.100941,1726
3,without_completed,533039,462387,70652,20973,0.328952,0.152298,0.542991,0.100194,1776
4,without_planned,412523,341871,70652,17329,0.328130,0.142047,0.539906,0.097109,1738


#### Analysis

The leave-one-out ablation shows that the individual status signals contribute very differently to ALS candidate generation.

Removing either `watching` or `planned` causes the largest decline in hybrid candidate recall, reducing union recall by approximately 0.0041. Both signals therefore provide important information despite representing different stages of user intent.

Removing `completed` has a smaller but still measurable negative effect, while removing `rewatching` changes union recall only slightly. The latter result should be interpreted cautiously because rewatching interactions are relatively rare in the training data.

These results also show that grouping `completed` and `planned` under a shared confidence is not well supported: `planned` has a substantially larger marginal effect on candidate quality.

All four non-zero status signals are retained, but the weights of `watching`, `planned`, and `completed` are evaluated separately in the following experiments. The `rewatching` confidence is kept at its baseline value because the available signal is too sparse to justify fine-grained tuning.

### Watching confidence

The leave-one-out ablation identifies `watching` as one of the most influential status signals.

Its confidence is varied independently while all other interaction weights remain fixed. This evaluates whether the baseline value of 1.0 assigns an appropriate relative strength to active watching behavior.

In [10]:
watching_values = [0.25, 0.5, 1.0, 1.5, 2.0]

als_watching_results = []

for watching_confidence in watching_values:
    signal_confidences = baseline_confidences.model_copy(
        update={
            "watching": watching_confidence,
        }
    )

    result = evaluate_als_variant(
        signal_confidences=signal_confidences,
    )

    als_watching_results.append(
        {
            "watching_confidence": watching_confidence,
            **result,
        }
    )

als_watching_results = pd.DataFrame(als_watching_results)

als_watching_results

,watching_confidence,training_signals,positive_signals,negative_signals,supported_items,als_recall_at_100,als_catalog_coverage_at_100,union_recall,als_marginal_recall,als_unique_relevant_hits
0,0.25,629576,558924,70652,21619,0.326298,0.157382,0.540685,0.097888,1716
1,0.50,629576,558924,70652,21619,0.328459,0.158138,0.542996,0.100199,1722
2,1.00,629576,558924,70652,21619,0.331908,0.158936,0.543957,0.101160,1742
3,1.50,629576,558924,70652,21619,0.332279,0.158936,0.543591,0.100794,1739
4,2.00,629576,558924,70652,21619,0.332864,0.159314,0.544041,0.101244,1739


#### Selection

The `watching` signal has a clear positive effect, but its exact confidence is not sharply identified.

Increasing the confidence from 0.25 to 1.0 consistently improves both standalone ALS retrieval and hybrid candidate recall. Values above 1.0 provide no consistent additional benefit.

A confidence of 2.0 produces the numerically highest union recall, but the improvement over the baseline value of 1.0 is only 0.00008 and is accompanied by a slightly lower number of uniquely retrieved relevant pairs. Given the small and non-monotonic differences, this is not sufficient evidence to prefer the larger weight.

The baseline `watching_confidence = 1.0` is therefore retained.

### Planned confidence

The leave-one-out ablation identifies `planned` as one of the most influential status signals despite its baseline confidence being lower than that of `watching`.

Its confidence is therefore varied independently while all other interaction weights remain fixed. This experiment evaluates whether planned interactions should remain a relatively weak positive signal or receive greater weight.

In [11]:
planned_values = [0.0, 0.25, 0.5, 0.75, 1.0]

als_planned_results = []

for planned_confidence in planned_values:
    signal_confidences = baseline_confidences.model_copy(
        update={
            "planned": planned_confidence,
        }
    )

    result = evaluate_als_variant(
        signal_confidences=signal_confidences,
    )

    als_planned_results.append(
        {
            "planned_confidence": planned_confidence,
            **result,
        }
    )

als_planned_results = pd.DataFrame(als_planned_results)

als_planned_results

,planned_confidence,training_signals,positive_signals,negative_signals,supported_items,als_recall_at_100,als_catalog_coverage_at_100,union_recall,als_marginal_recall,als_unique_relevant_hits
0,0.00,412523,341871,70652,17329,0.328130,0.142047,0.539906,0.097109,1738
1,0.25,629576,558924,70652,21619,0.332131,0.152802,0.544640,0.101843,1751
2,0.50,629576,558924,70652,21619,0.331908,0.158936,0.543957,0.101160,1742
3,0.75,629576,558924,70652,21619,0.326936,0.163642,0.541218,0.098421,1727
4,1.00,629576,558924,70652,21619,0.325510,0.167591,0.542352,0.099555,1723


#### Selection

The `planned` signal provides useful information, but its contribution depends strongly on its relative confidence.

Removing planned interactions substantially reduces hybrid candidate recall, confirming that the signal should be retained. However, increasing its weight beyond a relatively weak positive confidence progressively shifts the candidate distribution without improving relevant-item retrieval.

A confidence of 0.25 achieves the highest standalone ALS Recall@100, union recall, marginal recall contribution, and number of uniquely retrieved relevant user-item pairs among the tested values.

Therefore, `planned_confidence = 0.25` is selected for subsequent experiments. This value represents the best tested trade-off rather than a precisely identified global optimum.

In [12]:
selected_status_confidences = baseline_confidences.model_copy(
    update={
        "watching": 1.0,
        "planned": 0.25,
    }
)

### Completed confidence

The leave-one-out ablation shows that `completed` interactions provide a measurable but smaller contribution than `watching` and `planned`.

Its confidence is varied independently using the status configuration selected in the previous experiments. This evaluates whether completed interactions should remain a moderate positive signal or receive weaker or stronger weighting.

In [13]:
completed_values = [0.0, 0.25, 0.5, 0.75, 1.0]

als_completed_results = []

for completed_confidence in completed_values:
    signal_confidences = selected_status_confidences.model_copy(
        update={
            "completed": completed_confidence,
        }
    )

    result = evaluate_als_variant(
        signal_confidences=signal_confidences,
    )

    als_completed_results.append(
        {
            "completed_confidence": completed_confidence,
            **result,
        }
    )

als_completed_results = pd.DataFrame(als_completed_results)

als_completed_results

,completed_confidence,training_signals,positive_signals,negative_signals,supported_items,als_recall_at_100,als_catalog_coverage_at_100,union_recall,als_marginal_recall,als_unique_relevant_hits
0,0.00,533039,462387,70652,20973,0.334508,0.144736,0.544962,0.102165,1751
1,0.25,629576,558924,70652,21619,0.338244,0.149315,0.548324,0.105527,1778
2,0.50,629576,558924,70652,21619,0.332131,0.152802,0.544640,0.101843,1751
3,0.75,629576,558924,70652,21619,0.327151,0.156247,0.540869,0.098072,1710
4,1.00,629576,558924,70652,21619,0.329861,0.156163,0.541614,0.098817,1735


#### Selection

The `completed` signal benefits ALS candidate generation, but the baseline confidence of 0.5 appears stronger than necessary.

With the previously selected `planned_confidence = 0.25`, reducing the completed confidence to 0.25 produces the highest standalone ALS Recall@100, union recall, marginal recall contribution, and number of uniquely retrieved relevant user-item pairs among the tested values.

Removing completed interactions entirely also performs better than the original confidence of 0.5, indicating that excessive weighting of this signal can hurt candidate quality.

Therefore, `completed_confidence = 0.25` is selected for subsequent experiments.

In [14]:
selected_status_confidences = selected_status_confidences.model_copy(
    update={
        "completed": 0.25,
    }
)

### On-hold signal

The baseline configuration assigns zero confidence to `on_hold` interactions, effectively excluding them from ALS training.

Unlike active watching or completed interactions, an on-hold status is ambiguous: it indicates that the user engaged with an item but temporarily stopped consuming it. The signal may therefore carry weak positive preference information, but assigning too much confidence could introduce noise.

The `on_hold` confidence is evaluated using the status weights selected in the previous experiments.

In [15]:
on_hold_values = [0.0, 0.1, 0.25, 0.5, 1.0]

als_on_hold_results = []

for on_hold_confidence in on_hold_values:
    signal_confidences = selected_status_confidences.model_copy(
        update={
            "on_hold": on_hold_confidence,
        }
    )

    result = evaluate_als_variant(
        signal_confidences=signal_confidences,
    )

    als_on_hold_results.append(
        {
            "on_hold_confidence": on_hold_confidence,
            **result,
        }
    )

als_on_hold_results = pd.DataFrame(als_on_hold_results)

als_on_hold_results

,on_hold_confidence,training_signals,positive_signals,negative_signals,supported_items,als_recall_at_100,als_catalog_coverage_at_100,union_recall,als_marginal_recall,als_unique_relevant_hits
0,0.00,629576,558924,70652,21619,0.338244,0.149315,0.548324,0.105527,1778
1,0.10,641753,571101,70652,21631,0.336045,0.149945,0.547015,0.104218,1773
2,0.25,641753,571101,70652,21631,0.334284,0.150408,0.545647,0.102850,1762
3,0.50,641753,571101,70652,21631,0.333788,0.150954,0.545675,0.102878,1762
4,1.00,641753,571101,70652,21631,0.330291,0.152046,0.544265,0.101468,1757


#### Selection

Treating `on_hold` interactions as a positive preference signal does not improve ALS candidate generation.

Even a weak positive confidence of 0.10 reduces standalone ALS Recall@100, union recall, marginal recall contribution, and the number of uniquely retrieved relevant user-item pairs compared with excluding the signal entirely.

Increasing the confidence further generally weakens candidate quality while producing only a small increase in catalog coverage.

Therefore, `on_hold_confidence = 0.0` is retained, and on-hold interactions are excluded from the ALS training representation.

### Negative signal strength

The signal-composition ablation showed that explicit negative ratings provide a smaller additional gain after positive ratings and status interactions are included.

To evaluate their strength without changing their relative meaning, both negative-rating confidences are scaled by a common multiplier. Status and positive-rating confidences remain fixed at their selected values.

In [16]:
negative_multipliers = [0.0, 0.5, 1.0, 1.5, 2.0]

als_negative_strength_results = []

for multiplier in negative_multipliers:
    signal_confidences = selected_status_confidences.model_copy(
        update={
            "rating_4_5": baseline_confidences.rating_4_5 * multiplier,
            "rating_1_3": baseline_confidences.rating_1_3 * multiplier,
        }
    )

    result = evaluate_als_variant(
        signal_confidences=signal_confidences,
    )

    als_negative_strength_results.append(
        {
            "negative_multiplier": multiplier,
            "rating_4_5_confidence": signal_confidences.rating_4_5,
            "rating_1_3_confidence": signal_confidences.rating_1_3,
            **result,
        }
    )

als_negative_strength_results = pd.DataFrame(als_negative_strength_results)

als_negative_strength_results

,negative_multiplier,rating_4_5_confidence,rating_1_3_confidence,training_signals,positive_signals,negative_signals,supported_items,als_recall_at_100,als_catalog_coverage_at_100,union_recall,als_marginal_recall,als_unique_relevant_hits
0,0.0,-0.0,-0.0,558924,558924,0,21534,0.332906,0.149105,0.546096,0.103299,1764
1,0.5,-0.5,-1.0,629576,558924,70652,21619,0.337910,0.149735,0.547585,0.104788,1794
2,1.0,-1.0,-2.0,629576,558924,70652,21619,0.338244,0.149315,0.548324,0.105527,1778
3,1.5,-1.5,-3.0,629576,558924,70652,21619,0.337491,0.148895,0.547729,0.104932,1773
4,2.0,-2.0,-4.0,629576,558924,70652,21619,0.333743,0.149231,0.545616,0.102819,1771


#### Selection

Explicit negative-rating signals provide a measurable improvement after the selected status weights are applied.

Removing negative signals reduces union recall from 0.5483 to 0.5461. A multiplier of 1.0 achieves the highest standalone ALS Recall@100, union recall, and marginal recall contribution among the tested values.

A multiplier of 0.5 retrieves a slightly larger raw number of unique relevant user-item pairs, but this does not translate into higher macro recall across users.

The baseline overall negative-signal strength is therefore retained, corresponding to confidences of `-1.0` for ratings 4–5 and `-2.0` for ratings 1–3.

### Negative rating severity

The previous experiment confirms that negative ratings are useful as a group, but the relative weighting between moderately low ratings (4–5) and strongly negative ratings (1–3) remains heuristic.

A small set of severity schemes is compared while the selected status configuration and overall ALS hyperparameters remain fixed. This evaluates whether strongly negative ratings should receive greater weight than moderately negative feedback.

In [17]:
negative_severity_variants = {
    "equal": {
        "rating_4_5": -1.0,
        "rating_1_3": -1.0,
    },
    "baseline_1_to_2": {
        "rating_4_5": -1.0,
        "rating_1_3": -2.0,
    },
    "stronger_separation": {
        "rating_4_5": -1.0,
        "rating_1_3": -3.0,
    },
    "weaker_moderate_negative": {
        "rating_4_5": -0.5,
        "rating_1_3": -2.0,
    },
}

als_negative_severity_results = []

for variant_name, updates in negative_severity_variants.items():
    signal_confidences = selected_status_confidences.model_copy(update=updates)

    result = evaluate_als_variant(
        signal_confidences=signal_confidences,
    )

    als_negative_severity_results.append(
        {
            "variant": variant_name,
            "rating_4_5_confidence": signal_confidences.rating_4_5,
            "rating_1_3_confidence": signal_confidences.rating_1_3,
            **result,
        }
    )

als_negative_severity_results = pd.DataFrame(als_negative_severity_results)

als_negative_severity_results

,variant,rating_4_5_confidence,rating_1_3_confidence,training_signals,positive_signals,negative_signals,supported_items,als_recall_at_100,als_catalog_coverage_at_100,union_recall,als_marginal_recall,als_unique_relevant_hits
0,equal,-1.0,-1.0,629576,558924,70652,21619,0.338288,0.149819,0.547904,0.105107,1784
1,baseline_1_to_2,-1.0,-2.0,629576,558924,70652,21619,0.338244,0.149315,0.548324,0.105527,1778
2,stronger_separation,-1.0,-3.0,629576,558924,70652,21619,0.336442,0.149147,0.546821,0.104024,1769
3,weaker_moderate_negative,-0.5,-2.0,629576,558924,70652,21619,0.336985,0.149819,0.547876,0.105079,1781


#### Selection

The relative strength of negative ratings has only a small effect within the tested range.

Treating ratings 1–3 as a stronger negative signal than ratings 4–5 produces the highest hybrid candidate recall. The baseline `-1.0 / -2.0` scheme achieves the highest union recall and marginal recall contribution, although the differences from equal negative weighting are small.

Stronger separation between the two negative-rating groups reduces candidate quality, while weakening moderately negative ratings provides no clear benefit.

The baseline confidences of `-1.0` for ratings 4–5 and `-2.0` for ratings 1–3 are therefore retained. The result should be interpreted as support for the existing relative weighting rather than evidence of a sharply identified optimum.

In [18]:
selected_signal_confidences = baseline_confidences.model_copy(
    update={
        "watching": 1.0,
        "rewatching": 1.0,
        "completed": 0.25,
        "planned": 0.25,
        "on_hold": 0.0,
        "rating_4_5": -1.0,
        "rating_1_3": -2.0,
    }
)

## Model hyperparameter experiments

With the interaction representation and relative signal weights selected, the following experiments tune the ALS model itself.

The selected signal configuration is kept fixed while model capacity, regularization, optimization duration, and overall confidence scaling are evaluated.

### Latent factor dimensionality

The number of latent factors controls the capacity of the ALS representation.

Higher dimensionality can capture more complex user-item preference structure, but increases model size, training cost, and inference computation approximately linearly.

The current baseline uses 96 latent factors.

In [19]:
factor_values = [32, 64, 96, 128, 192]

als_factor_results = []

for factors in factor_values:
    result = evaluate_als_variant(
        signal_confidences=selected_signal_confidences,
        factors=factors,
    )

    als_factor_results.append(
        {
            "factors": factors,
            **result,
        }
    )

als_factor_results = pd.DataFrame(als_factor_results)

als_factor_results

,factors,training_signals,positive_signals,negative_signals,supported_items,als_recall_at_100,als_catalog_coverage_at_100,union_recall,als_marginal_recall,als_unique_relevant_hits
0,32,629576,558924,70652,21619,0.349787,0.124821,0.534186,0.091389,1690
1,64,629576,558924,70652,21619,0.345621,0.141333,0.542656,0.099859,1783
2,96,629576,558924,70652,21619,0.338244,0.149315,0.548324,0.105527,1778
3,128,629576,558924,70652,21619,0.317665,0.156457,0.541888,0.099091,1744
4,192,629576,558924,70652,21619,0.292566,0.161667,0.539941,0.097144,1639


#### Selection

Latent dimensionality strongly affects the role of ALS within the hybrid candidate generator.

Lower-dimensional models achieve higher standalone ALS Recall@100, but retrieve a narrower set of candidates and provide substantially weaker complementary value to the hybrid pool.

Increasing the dimensionality from 64 to 96 factors improves union recall from 0.5427 to 0.5483 and marginal recall contribution from 0.0999 to 0.1055. Increasing the dimensionality further to 128 or 192 factors reduces hybrid candidate quality despite increasing catalog coverage.

The baseline value of `factors = 96` is therefore retained. It provides the strongest hybrid recall among the tested configurations without requiring the additional model size and computation of higher-dimensional variants.

### Regularization

Regularization controls the penalty applied to the learned ALS factors.

Insufficient regularization may allow the latent representation to fit interaction noise, while excessive regularization can suppress useful preference structure. The selected interaction representation and latent dimensionality of 96 factors are kept fixed during this experiment.

In [20]:
regularization_values = [0.1, 0.2, 0.5, 1.0, 2.0]

als_regularization_results = []

for regularization in regularization_values:
    result = evaluate_als_variant(
        signal_confidences=selected_signal_confidences,
        factors=96,
        regularization=regularization,
    )

    als_regularization_results.append(
        {
            "regularization": regularization,
            **result,
        }
    )

als_regularization_results = pd.DataFrame(als_regularization_results)

als_regularization_results

,regularization,training_signals,positive_signals,negative_signals,supported_items,als_recall_at_100,als_catalog_coverage_at_100,union_recall,als_marginal_recall,als_unique_relevant_hits
0,0.1,629576,558924,70652,21619,0.338244,0.149315,0.548324,0.105527,1778
1,0.2,629576,558924,70652,21619,0.338951,0.148979,0.548121,0.105324,1783
2,0.5,629576,558924,70652,21619,0.340465,0.148013,0.548802,0.106005,1794
3,1.0,629576,558924,70652,21619,0.341919,0.145954,0.548808,0.106011,1796
4,2.0,629576,558924,70652,21619,0.340834,0.145114,0.548043,0.105246,1811


#### Selection

Increasing regularization from the baseline value of 0.10 improves ALS candidate quality, with performance reaching a plateau around 0.5–1.0.

Regularization values of 0.5 and 1.0 produce effectively identical hybrid results: their union recall and marginal recall contribution differ by less than 0.00001. Increasing regularization further to 2.0 reduces both metrics despite producing slightly more unique relevant hits.

Since the validation results do not provide meaningful evidence that 1.0 is better than 0.5, `regularization = 0.5` is selected as the lower end of the observed performance plateau.

This value will be reconsidered after tuning the overall confidence scaling because ALS regularization and confidence magnitude are not fully independent.

### Confidence scaling

The ALS `alpha` parameter controls the overall scale of interaction confidence while preserving the relative weighting between the selected signals.

Changing the confidence scale affects the balance between fitting observed interactions and regularization. The selected signal representation, 96 latent factors, and regularization of 0.5 are kept fixed during this experiment.

In [21]:
alpha_values = [0.25, 0.5, 1.0, 2.0, 4.0]

als_alpha_results = []

for alpha in alpha_values:
    result = evaluate_als_variant(
        signal_confidences=selected_signal_confidences,
        factors=96,
        regularization=0.5,
        alpha=alpha,
    )

    als_alpha_results.append(
        {
            "alpha": alpha,
            **result,
        }
    )

als_alpha_results = pd.DataFrame(als_alpha_results)

als_alpha_results

,alpha,training_signals,positive_signals,negative_signals,supported_items,als_recall_at_100,als_catalog_coverage_at_100,union_recall,als_marginal_recall,als_unique_relevant_hits
0,0.25,629576,558924,70652,21619,0.325558,0.135577,0.536519,0.093722,1762
1,0.50,629576,558924,70652,21619,0.333505,0.143391,0.542089,0.099292,1795
2,1.00,629576,558924,70652,21619,0.340465,0.148013,0.548802,0.106005,1794
3,2.00,629576,558924,70652,21619,0.333877,0.154819,0.544602,0.101805,1747
4,4.00,629576,558924,70652,21619,0.323151,0.159230,0.538410,0.095613,1673


#### Selection

The overall confidence scale has a pronounced effect on ALS candidate quality.

Increasing `alpha` from 0.25 to 1.0 consistently improves standalone ALS recall and the retriever's contribution to the hybrid candidate pool. At `alpha = 1.0`, union recall reaches 0.5488 and marginal recall contribution reaches 0.1060, both the highest values among the tested configurations.

Larger confidence scales increase catalog coverage but substantially reduce relevant candidate retrieval. A lower value of 0.5 produces one additional unique relevant pair, but this does not compensate for its considerably lower macro recall.

Therefore, the baseline `alpha = 1.0` is retained.

### Training iterations

The number of ALS iterations controls how long the alternating optimization procedure updates user and item factors.

Too few iterations may leave the factorization insufficiently optimized, while additional iterations increase offline training cost and may provide diminishing validation gains.

The selected signal representation, 96 latent factors, regularization of 0.5, and `alpha = 1.0` are kept fixed.

In [22]:
iteration_values = [10, 20, 40, 60, 80]

als_iteration_results = []

for iterations in iteration_values:
    result = evaluate_als_variant(
        signal_confidences=selected_signal_confidences,
        factors=96,
        regularization=0.5,
        alpha=1.0,
        iterations=iterations,
    )

    als_iteration_results.append(
        {
            "iterations": iterations,
            **result,
        }
    )

als_iteration_results = pd.DataFrame(als_iteration_results)

als_iteration_results

,iterations,training_signals,positive_signals,negative_signals,supported_items,als_recall_at_100,als_catalog_coverage_at_100,union_recall,als_marginal_recall,als_unique_relevant_hits
0,10,629576,558924,70652,21619,0.333184,0.147215,0.542164,0.099367,1785
1,20,629576,558924,70652,21619,0.340870,0.147425,0.548511,0.105714,1786
2,40,629576,558924,70652,21619,0.340465,0.148013,0.548802,0.106005,1794
3,60,629576,558924,70652,21619,0.335752,0.148391,0.546147,0.103350,1773
4,80,629576,558924,70652,21619,0.334312,0.148475,0.544756,0.101958,1760


#### Selection

ALS candidate quality improves substantially when the number of training iterations is increased from 10 to 20.

Increasing the training duration further from 20 to 40 iterations produces only a small additional gain in hybrid candidate recall: union recall increases by approximately 0.0003, while the offline optimization cost roughly doubles.

Longer training at 60 and 80 iterations reduces both standalone and hybrid retrieval quality, indicating that additional optimization does not provide a consistent benefit.

Therefore, `iterations = 20` is selected as the current training-cost and candidate-quality trade-off.

## Final ALS configuration

The experiments selected both a revised interaction representation and a set of ALS model hyperparameters.

Before updating the training configuration, the complete selected setup is compared directly with the original ALS baseline. This verifies that the improvements observed during sequential tuning are preserved when all selected changes are applied together.

In [23]:
baseline_evaluation = evaluate_retriever_run(
    baseline_run,
    candidate_count=candidate_count,
    catalog_size=len(catalog),
)

baseline_als_metrics = baseline_evaluation.individual_metrics.set_index("source").loc[
    RetrieverName.IMPLICIT_ALS.value
]

baseline_als_contribution = next(
    contribution
    for contribution in baseline_evaluation.contributions
    if contribution.source == RetrieverName.IMPLICIT_ALS
)

selected_als_result = evaluate_als_variant(
    signal_confidences=selected_signal_confidences,
    factors=96,
    regularization=0.5,
    alpha=1.0,
    iterations=20,
)

als_final_comparison = pd.DataFrame(
    [
        {
            "configuration": "baseline",
            "als_recall_at_100": baseline_als_metrics["recall_at_k"],
            "als_catalog_coverage_at_100": (
                baseline_als_metrics["catalog_coverage_at_k"]
            ),
            "union_recall": baseline_evaluation.union_recall,
            "als_marginal_recall": (baseline_als_contribution.marginal_recall),
            "als_unique_relevant_hits": (
                baseline_als_contribution.unique_relevant_hits
            ),
        },
        {
            "configuration": "selected",
            "als_recall_at_100": selected_als_result["als_recall_at_100"],
            "als_catalog_coverage_at_100": (
                selected_als_result["als_catalog_coverage_at_100"]
            ),
            "union_recall": selected_als_result["union_recall"],
            "als_marginal_recall": (selected_als_result["als_marginal_recall"]),
            "als_unique_relevant_hits": (
                selected_als_result["als_unique_relevant_hits"]
            ),
        },
    ]
)

als_final_comparison

,configuration,als_recall_at_100,als_catalog_coverage_at_100,union_recall,als_marginal_recall,als_unique_relevant_hits
0,baseline,0.331908,0.158936,0.543957,0.101160,1742
1,selected,0.340870,0.147425,0.548511,0.105714,1786


## Conclusion

The ALS experiments evaluated both the interaction representation and the main model hyperparameters on the validation period while keeping the final test split untouched.

The selected interaction confidences are:

- `rating_8_10 = 2.0`
- `watching = 1.0`
- `rewatching = 1.0`
- `completed = 0.25`
- `planned = 0.25`
- `on_hold = 0.0`
- `rating_4_5 = -1.0`
- `rating_1_3 = -2.0`

The selected ALS hyperparameters are:

- `factors = 96`
- `regularization = 0.5`
- `alpha = 1.0`
- `iterations = 20`

Status-based interactions provide substantial complementary information beyond explicit ratings. `watching` and `planned` have the largest individual impact on hybrid candidate recall, while `completed` provides a smaller but measurable contribution. `on_hold` does not improve candidate quality when treated as a positive signal and is therefore excluded.

Reducing the confidences of `completed` and `planned` from 0.5 to 0.25 improves hybrid retrieval, while the baseline negative-rating scheme of `-1.0 / -2.0` remains effective.

For the ALS model itself, 96 latent factors provide the strongest hybrid candidate recall among the tested dimensionalities. Increasing regularization from 0.1 to 0.5 improves candidate quality, while the baseline `alpha = 1.0` remains the strongest tested confidence scale.

Twenty training iterations are selected instead of the original 40. Doubling the training duration from 20 to 40 provides only a small validation gain, while approximately doubling the corresponding offline optimization work.

When all selected changes are applied together, ALS Recall@100 increases from 0.3319 to 0.3409 and hybrid union recall increases from 0.5440 to 0.5485. The number of uniquely retrieved relevant user-item pairs also increases from 1,742 to 1,786. Catalog coverage decreases from 0.1589 to 0.1474, representing a trade-off in favor of more relevant and complementary candidate retrieval.

The selected configuration therefore improves the primary candidate-quality metrics while reducing ALS training iterations. Its effect on the complete recommendation pipeline will be evaluated later using end-to-end ranking metrics.